In [ ]:
# %pip install requests geopandas shapely fiona


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import geopandas as gpd
from shapely.geometry import shape
import geopandas as gpd

In [5]:
bbox = "172.5,-43.7,173.2,-43.3"  # minLon, minLat, maxLon, maxLat

timestamp = "2018-01-01T00:00:00Z"

url = "https://api.ohsome.org/v1/elements/geometry"

params = {
    "bboxes": bbox,
    "time": timestamp,
    "types": "way",
    "keys": "highway" 
}

response = requests.get(url, params=params)
data = response.json()

# Convert to GeoDataFrame
features = []
for feat in data["features"]:
    geom = shape(feat["geometry"])
    tags = feat["properties"]
    features.append({"geometry": geom, **tags})

roads_2018 = gpd.GeoDataFrame(features, crs="EPSG:4326").to_crs('EPSG:2193')

roads_2018


,geometry,@osmId,@snapshotTimestamp,highway
0,"LINESTRING (1567504.34 5186337.689, 1567451.64...",way/4840181,2018-01-01T00:00:00Z,tertiary
1,"LINESTRING (1565831.928 5184057.631, 1565910.4...",way/4840188,2018-01-01T00:00:00Z,secondary
2,"LINESTRING (1565876.536 5183070.457, 1565846.0...",way/4874214,2018-01-01T00:00:00Z,footway
3,"LINESTRING (1567428.726 5185438.19, 1567396.38...",way/4925131,2018-01-01T00:00:00Z,residential
4,"LINESTRING (1567154.076 5184951.688, 1567191.7...",way/4925134,2018-01-01T00:00:00Z,residential
...,...,...,...,...
17198,"LINESTRING (1578933.858 5169773.401, 1578930.5...",way/539827235,2018-01-01T00:00:00Z,residential
17199,"LINESTRING (1578616.902 5168616.799, 1578624.7...",way/539827236,2018-01-01T00:00:00Z,unclassified
17200,"LINESTRING (1578624.778 5168603.131, 1578634.5...",way/539827237,2018-01-01T00:00:00Z,unclassified
17201,"LINESTRING (1578623.935 5168665.067, 1578620.7...",way/539827238,2018-01-01T00:00:00Z,footway


In [ ]:
# roads_2018.to_file("output/chc_roads.gpkg", driver="GPKG")

In [7]:
roads = gpd.read_file('output/chc_roads.gpkg')
ta = gpd.read_file('data/chc-boundaries/territorial-authority-2021-generalised.gpkg', engine='pyogrio')
sa2 = gpd.read_file('data/chc-boundaries/sa2/statistical-area-2-2023-generalised.shp')

In [8]:
roads.crs

<Projected CRS: EPSG:2193>
Name: NZGD2000 / New Zealand Transverse Mercator 2000
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: New Zealand - North Island, South Island, Stewart Island - onshore.
- bounds: (166.37, -47.33, 178.63, -34.1)
Coordinate Operation:
- name: New Zealand Transverse Mercator 2000
- method: Transverse Mercator
Datum: New Zealand Geodetic Datum 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [22]:
ta_chc = ta[ta['TA2021_V1_00_NAME_ASCII'] == 'Christchurch City']
sa2_chc = gpd.clip(sa2, ta_chc)
boundary = sa2_chc[sa2_chc['SA22023__2'].str.lower().str.contains('fendalton')]

In [23]:
roads_chc = gpd.clip(roads, boundary)

In [24]:
roads_chc.explore()